In [ ]:
# CELL 1
from google.colab import drive
try:
  drive.mount('/content/gdrive')
except:
  print("Load the drive manually from the left panel first. Then run this again,")
  pathBase="/content/drive/MyDrive/"

Mounted at /content/gdrive


In [ ]:
# CELL 2
#Import necessary Python libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
# CELL 3
# make sure you load all csv file to the /content/gdrive/MyDrive/158736/Assignment_1 folder. This will help us to run the code in our Colab environemnt, without having to change the dataset location in individual Colab notebooks.
file_path = '/content/gdrive/MyDrive/158736/Assignment_1/cow_accelerometer_data.xlsx'
df = pd.read_excel(file_path)

In [ ]:
# CELL 4
# i. Split dataset into train, validation and test subsets
# We use GroupShuffleSplit on 'session' to ensure sequential time-series data isn't leaked across splits
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['session']))
train_data = df.iloc[train_idx]
temp_data = df.iloc[temp_idx]

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_data, groups=temp_data['session']))
val_data = temp_data.iloc[val_idx]
test_data = temp_data.iloc[test_idx]

X_train = train_data.drop(columns=['session', 'animal', 'behavior'])
y_train = train_data['behavior']
X_val = val_data.drop(columns=['session', 'animal', 'behavior'])
y_val = val_data['behavior']
X_test = test_data.drop(columns=['session', 'animal', 'behavior'])
y_test = test_data['behavior']

# Print the number of samples in each set
print(f"Training data size: {len(X_train)} rows")
print(f"Validation data size: {len(X_val)} rows")
print(f"Test data size: {len(X_test)} rows")


Training data size: 8745 rows
Validation data size: 1463 rows
Test data size: 1111 rows


In [ ]:
# CELL 5
# ii. Data normalization, if needed
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)

# Pick 3 columns to look at as an example
cols_to_view = ['acc_x_mean', 'acc_x_sum', 'odba']

print("--- BEFORE NORMALIZATION ---")
print("First 3 rows of data:")
print(X_train[cols_to_view].head(3))
print("\nAverage (Mean) values:")
print(X_train[cols_to_view].mean().round(4))
print("\nSpread (Standard Deviation):")
print(X_train[cols_to_view].std().round(4))

print("\n----------------------------------\n")

print("--- AFTER NORMALIZATION ---")
print("First 3 rows of data:")
print(X_train_scaled_df[cols_to_view].head(3))
print("\nAverage (Mean) values:")
print(X_train_scaled_df[cols_to_view].mean().round(4))
print("\nSpread (Standard Deviation):")
print(X_train_scaled_df[cols_to_view].std().round(4))

#iii. Handling the imbalance issue, if needed
# We calculate class weights to penalize the model more for missing the minority classes
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

--- BEFORE NORMALIZATION ---
First 3 rows of data:
   acc_x_mean  acc_x_sum     odba
0    -0.01008     -0.252  0.07080
1     0.02640      0.660  0.09303
2     0.07264      1.816  0.11415

Average (Mean) values:
acc_x_mean   -0.0753
acc_x_sum    -1.8837
odba          0.0729
dtype: float64

Spread (Standard Deviation):
acc_x_mean    0.1612
acc_x_sum     4.0297
odba          0.0870
dtype: float64

----------------------------------

--- AFTER NORMALIZATION ---
First 3 rows of data:
   acc_x_mean  acc_x_sum      odba
0    0.404951   0.404951 -0.023601
1    0.631282   0.631282  0.232048
2    0.918166   0.918166  0.474932

Average (Mean) values:
acc_x_mean    0.0
acc_x_sum     0.0
odba          0.0
dtype: float64

Spread (Standard Deviation):
acc_x_mean    1.0001
acc_x_sum     1.0001
odba          1.0001
dtype: float64


In [ ]:
# CELL 6
#iv - vi implement the classifiers and evalaute separately. For evaluation, use classificationreport functionality of sklearn
print("--- Evaluating Classifiers ---")

# Multi-Layer Perceptron
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp.fit(X_train_scaled, y_train)
print("MLP Results:")
print(classification_report(y_test, mlp.predict(X_test_scaled)))

# Support Vector Machine (with class weights)
svm_model = SVC(class_weight=class_weights, random_state=42)
svm_model.fit(X_train_scaled, y_train)
print("\nSVM Results:")
print(classification_report(y_test, svm_model.predict(X_test_scaled)))

# Random Forest (with class weights)
rf_model = RandomForestClassifier(class_weight=class_weights, random_state=42)
rf_model.fit(X_train_scaled, y_train)
print("\nRandom Forest Results:")
print(classification_report(y_test, rf_model.predict(X_test_scaled)))

--- Evaluating Classifiers ---
MLP Results:
              precision    recall  f1-score   support

     feeding       0.88      0.94      0.91        49
      moving       0.90      0.83      0.86       158
     resting       0.97      0.92      0.95       808
  ruminating       0.60      0.93      0.73        96

    accuracy                           0.91      1111
   macro avg       0.84      0.90      0.86      1111
weighted avg       0.93      0.91      0.91      1111


SVM Results:
              precision    recall  f1-score   support

     feeding       0.88      0.94      0.91        49
      moving       0.89      0.91      0.90       158
     resting       0.99      0.90      0.94       808
  ruminating       0.56      0.94      0.70        96

    accuracy                           0.91      1111
   macro avg       0.83      0.92      0.86      1111
weighted avg       0.93      0.91      0.91      1111


Random Forest Results:
              precision    recall  f1-score   su

In [ ]:
# CELL 7
# vii Calculate correlation between features
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [column for column in upper.columns if any(upper[column] > 0.9)]
print(f"\nNumber of highly correlated features (>0.9): {len(high_corr)}")

#viii PCA and FDA
pca = PCA(n_components=0.95) # Retain 95% of variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
rf_pca = RandomForestClassifier(random_state=42).fit(X_train_pca, y_train)
print("\nRF with PCA Results:")
print(classification_report(y_test, rf_pca.predict(X_test_pca)))

lda = LDA() # FDA equivalent
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)
rf_lda = RandomForestClassifier(random_state=42).fit(X_train_lda, y_train)
print("\nRF with FDA Results:")
print(classification_report(y_test, rf_lda.predict(X_test_lda)))

#ix Feature selection
selector = SelectKBest(f_classif, k=20)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel = selector.transform(X_test_scaled)
rf_sel = RandomForestClassifier(random_state=42).fit(X_train_sel, y_train)
print("\nRF with SelectKBest (20 features) Results:")
print(classification_report(y_test, rf_sel.predict(X_test_sel)))


Number of highly correlated features (>0.9): 46

RF with PCA Results:
              precision    recall  f1-score   support

     feeding       0.88      0.88      0.88        49
      moving       0.91      0.84      0.87       158
     resting       0.95      0.95      0.95       808
  ruminating       0.67      0.81      0.74        96

    accuracy                           0.92      1111
   macro avg       0.85      0.87      0.86      1111
weighted avg       0.92      0.92      0.92      1111


RF with FDA Results:
              precision    recall  f1-score   support

     feeding       0.80      0.92      0.86        49
      moving       0.90      0.84      0.87       158
     resting       0.97      0.92      0.94       808
  ruminating       0.58      0.85      0.69        96

    accuracy                           0.90      1111
   macro avg       0.81      0.88      0.84      1111
weighted avg       0.92      0.90      0.91      1111


RF with SelectKBest (20 features) Re